# 04 — Explore visually

Quick exploration of the story before the publication chart, using the shared
Pillow `lollipop` template in both modes: single-value (lollipop) and two-value
(dumbbell). Charts render inline via `display()`.

> Note: this project renders with the shared **Pillow** factory rather than
> matplotlib. (matplotlib does not run in this project's Python 3.14 venv — a
> known `MarkerStyle` deepcopy recursion during axis-tick rendering — and the
> publication path is Pillow anyway, so matplotlib isn't a dependency here.)

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection
from chart_templates import lollipop
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
con = get_connection(cfg)
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15').df()
con.close()
films['label'] = films['title'] + '  (' + films['release_year'].astype(str) + ')'
img_w, img_h, _ = PRESETS['twitter_landscape']
def money(v):
    return f'${v/1e9:.2f}B' if v >= 1e9 else f'${v/1e6:.0f}M'
films[['title','adjusted_gross','nominal_gross','release_year']].head()

## Single-value lollipop — adjusted gross only
The ranked adjusted board on its own (lollipop mode).

In [ ]:
display(lollipop(
    films, category_col='label', value_col='adjusted_gross', value_fmt=money,
    title='Top 15 films by inflation-adjusted domestic gross',
    subtitle='Box Office Mojo, adjusted to 2022 dollars',
    dot_color='#005F73', img_width=img_w, img_height=img_h,
))

## Dumbbell — adjusted vs nominal
Adding `value2_col` turns it into a dumbbell: the gap between what a film made
at the time (gold) and its adjusted gross (teal) is the inflation effect.

In [ ]:
display(lollipop(
    films, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='Adjusted vs nominal domestic gross - top 15',
    subtitle='Teal = adjusted (2022 $), gold = nominal (release $)',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted', value2_label='Nominal', img_width=img_w, img_height=img_h,
))

---
**Next:** `06-viz-social.ipynb` builds the final publication chart (the same
dumbbell, with full titling/source) and saves it to `outputs/social/`.